In [1]:
# Generic inputs for most ML tasks
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
# This is new
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor

pd.options.display.float_format = '{:,.2f}'.format

# setup interactive notebook mode
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from IPython.display import display, HTML

In [2]:
# fetch data 

mco_syr_sw_data = pd.read_csv('flight_data/mco_syr_sw_combined.csv')
mco_syr_jb_data = pd.read_csv('flight_data/mco_syr_jb_combined.csv')
jfk_syr_jb_data = pd.read_csv('flight_data/jfk_syr_jb_combined.csv')
jfk_syr_dl_data = pd.read_csv('flight_data/jfk_syr_dl_combined.csv')
ord_syr_ua_data = pd.read_csv('flight_data/ord_syr_ua_combined.csv')
ord_syr_aa_data = pd.read_csv('flight_data/ord_syr_aa_combined.csv')

ord_syr_ua_data['dep_order'] = 'early'
jfk_syr_jb_data['dep_order'] = 'early'
mco_syr_sw_data['dep_order'] = 'early'
ord_syr_aa_data['dep_order'] = 'latter'
jfk_syr_dl_data['dep_order'] = 'latter'
mco_syr_jb_data['dep_order'] = 'latter'
#['dep_order'] = 'early'
# mco_syr_sw_data.head()
# mco_syr_jb_data.head()
# jfk_syr_jb_data.head()
# jfk_syr_dl_data.head()
# ord_syr_ua_data.head()
# ord_syr_aa_data.head()

In [3]:
dfs = [ord_syr_aa_data, ord_syr_ua_data, jfk_syr_dl_data, jfk_syr_jb_data, mco_syr_jb_data, mco_syr_sw_data]
main_data = pd.concat(dfs,axis = 0)
main_data.head()
len(main_data)

,Unnamed: 0,Carrier_Code,Date,Flight_Number,Tail_Number,Destination_Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),...,Wheels-on Time,Taxi-In time (Minutes),arr_Delay_Carrier,arr_Delay_Weather,arr_Delay_National_Aviation_System,arr_Delay_Security,arr_Delay_Late_Aircraft_Arrival,arr_hour,arr_day,arr_year
0,0,MQ,2020-01-04,"3,580.00",N240NN,SYR,7:55,8:21,112.00,87.00,...,10:43,5.00,0.00,0.00,0.00,0.00,0.00,10,5,2020
1,1,MQ,2020-01-11,"3,946.00",N247NN,SYR,15:00,15:09,108.00,132.00,...,18:13,8.00,9.00,0.00,24.00,0.00,0.00,17,5,2020
2,2,MQ,2020-01-18,"3,946.00",N265NN,SYR,15:00,16:27,108.00,147.00,...,19:45,9.00,0.00,4.00,39.00,0.00,83.00,17,5,2020
3,3,MQ,2020-01-25,"3,946.00",N281NN,SYR,15:00,14:55,108.00,126.00,...,17:56,5.00,0.00,0.00,0.00,0.00,0.00,17,5,2020
4,4,MQ,2020-02-01,"3,946.00",N283NN,SYR,15:00,14:57,108.00,99.00,...,17:32,4.00,0.00,0.00,0.00,0.00,0.00,17,5,2020


8661

In [4]:
main_data.dtypes
# main_data.dtypes['dep_min']



Unnamed: 0                              int64
Carrier_Code                           object
Date                                   object
Flight_Number                         float64
Tail_Number                            object
Destination_Airport                    object
Scheduled departure time               object
Actual departure time                  object
Scheduled elapsed time (Minutes)      float64
Actual elapsed time (Minutes)         float64
Departure delay (Minutes)             float64
Wheels-off time                        object
Taxi-Out time (Minutes)               float64
dep_Delay_Carrier                     float64
dep_Delay_Weather                     float64
dep_Delay_National_Aviation_System    float64
dep_Delay_Security                    float64
dep_Delay_Late_Aircraft_Arrival       float64
dep_hour                                int64
dep_day                                 int64
dep_year                                int64
dep_order                         

In [5]:
# def round_to_nearest_quarter(minute):
#     if 0 <= minute < 14:
#         return 0
#     elif 15 <= minute < 29:
#         return 15
#     elif 30 <= minute < 44:
#         return 30
#     else:
#         return 45
main_data['Date'] = pd.to_datetime( main_data['Date'],format ="%Y-%m-%d")
main_data['Date'] = main_data['Date'].dt.strftime('%m/%d/%Y')
main_data['dep_min'] = main_data['Scheduled departure time'].str.split(":").str[1].astype('int64')
# main_data['dep_minutes'] = main_data['dep_min'].apply(round_to_nearest_quarter).astype('object')
main_data['dep_min'] = main_data['dep_min'].astype('object')
main_data['arr_min'] = main_data['Scheduled Arrival Time'].str.split(":").str[1].astype('int64')
# main_data['arr_minutes'] = main_data['arr_min'].apply(round_to_nearest_quarter).astype('object')
main_data['arr_min'] = main_data['arr_min'].astype('object')
main_data['Flight_Number'] = main_data['Flight_Number'].astype('object')
main_data['dep_hours'] = main_data['dep_hour'].astype('object')
main_data['dep_hour'] = main_data['dep_hour'].astype('object')
main_data['dep_day'] = main_data['dep_day'].astype('object')
main_data['arr_hours'] = main_data['arr_hour'].astype('object')
main_data['arr_hour'] = main_data['arr_hour'].astype('object')
main_data['arr_day'] = main_data['arr_day'].astype('object')
conditions = [
    (main_data['Arrival Delay (Minutes)'] > 5),
    (main_data['Arrival Delay (Minutes)'] >= -5) & (main_data['Arrival Delay (Minutes)'] <= 5),
    (main_data['Arrival Delay (Minutes)'] < -5)
]
choices = [2, 1, 0]
main_data['arr_status'] = np.select(conditions, choices)
main_data.dtypes
main_data.head()

Unnamed: 0                              int64
Carrier_Code                           object
Date                                   object
Flight_Number                          object
Tail_Number                            object
Destination_Airport                    object
Scheduled departure time               object
Actual departure time                  object
Scheduled elapsed time (Minutes)      float64
Actual elapsed time (Minutes)         float64
Departure delay (Minutes)             float64
Wheels-off time                        object
Taxi-Out time (Minutes)               float64
dep_Delay_Carrier                     float64
dep_Delay_Weather                     float64
dep_Delay_National_Aviation_System    float64
dep_Delay_Security                    float64
dep_Delay_Late_Aircraft_Arrival       float64
dep_hour                               object
dep_day                                object
dep_year                                int64
dep_order                         

,Unnamed: 0,Carrier_Code,Date,Flight_Number,Tail_Number,Destination_Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),...,arr_Delay_Security,arr_Delay_Late_Aircraft_Arrival,arr_hour,arr_day,arr_year,dep_min,arr_min,dep_hours,arr_hours,arr_status
0,0,MQ,01/04/2020,"3,580.00",N240NN,SYR,7:55,8:21,112.00,87.00,...,0.00,0.00,10,5,2020,55,47,7,10,1
1,1,MQ,01/11/2020,"3,946.00",N247NN,SYR,15:00,15:09,108.00,132.00,...,0.00,0.00,17,5,2020,0,48,15,17,2
2,2,MQ,01/18/2020,"3,946.00",N265NN,SYR,15:00,16:27,108.00,147.00,...,0.00,83.00,17,5,2020,0,48,15,17,2
3,3,MQ,01/25/2020,"3,946.00",N281NN,SYR,15:00,14:55,108.00,126.00,...,0.00,0.00,17,5,2020,0,48,15,17,2
4,4,MQ,02/01/2020,"3,946.00",N283NN,SYR,15:00,14:57,108.00,99.00,...,0.00,0.00,17,5,2020,0,48,15,17,0


In [6]:
sub_data = main_data.drop(columns = ['Unnamed: 0','Destination_Airport','Actual departure time','Scheduled elapsed time (Minutes)','Actual elapsed time (Minutes)',
                         'Departure delay (Minutes)','Wheels-off time','Taxi-Out time (Minutes)','dep_Delay_Carrier', 'dep_year',
                         'dep_Delay_Weather','dep_Delay_National_Aviation_System','dep_Delay_Security','dep_Delay_Late_Aircraft_Arrival','Actual Arrival Time','Wheels-on Time','Taxi-In time (Minutes)','arr_Delay_Carrier','arr_Delay_Weather',
                         'arr_Delay_National_Aviation_System','arr_Delay_Security','arr_Delay_Late_Aircraft_Arrival','arr_year'])

In [7]:
sub_data.dtypes

Carrier_Code                 object
Date                         object
Flight_Number                object
Tail_Number                  object
Scheduled departure time     object
dep_hour                     object
dep_day                      object
dep_order                    object
Origin_Airport               object
Scheduled Arrival Time       object
Arrival Delay (Minutes)     float64
arr_hour                     object
arr_day                      object
dep_min                      object
arr_min                      object
dep_hours                    object
arr_hours                    object
arr_status                    int32
dtype: object

In [8]:
sub_data.head()

,Carrier_Code,Date,Flight_Number,Tail_Number,Scheduled departure time,dep_hour,dep_day,dep_order,Origin_Airport,Scheduled Arrival Time,Arrival Delay (Minutes),arr_hour,arr_day,dep_min,arr_min,dep_hours,arr_hours,arr_status
0,MQ,01/04/2020,"3,580.00",N240NN,7:55,7,5,latter,ORD,10:47,1.00,10,5,55,47,7,10,1
1,MQ,01/11/2020,"3,946.00",N247NN,15:00,15,5,latter,ORD,17:48,33.00,17,5,0,48,15,17,2
2,MQ,01/18/2020,"3,946.00",N265NN,15:00,15,5,latter,ORD,17:48,126.00,17,5,0,48,15,17,2
3,MQ,01/25/2020,"3,946.00",N281NN,15:00,15,5,latter,ORD,17:48,13.00,17,5,0,48,15,17,2
4,MQ,02/01/2020,"3,946.00",N283NN,15:00,15,5,latter,ORD,17:48,-12.00,17,5,0,48,15,17,0


In [9]:
latter_data = sub_data[sub_data['dep_order'] == 'latter']
len(latter_data)

5337

In [10]:
early_data = sub_data[sub_data['dep_order'] == 'early']
len(early_data)

3324

In [11]:
# latter_data['dep_year'].value_counts()

In [12]:
# early_data['dep_year'].value_counts()

In [13]:
# latter_data['Carrier_Code'].value_counts()

In [14]:
# early_data['Carrier_Code'].value_counts()

In [15]:
merged_df = pd.merge(latter_data, early_data, on=['Date', 'Origin_Airport'], how = 'left')
merged_df.head()
len(merged_df)

,Carrier_Code_x,Date,Flight_Number_x,Tail_Number_x,Scheduled departure time_x,dep_hour_x,dep_day_x,dep_order_x,Origin_Airport,Scheduled Arrival Time_x,...,dep_order_y,Scheduled Arrival Time_y,Arrival Delay (Minutes)_y,arr_hour_y,arr_day_y,dep_min_y,arr_min_y,dep_hours_y,arr_hours_y,arr_status_y
0,MQ,01/04/2020,"3,580.00",N240NN,7:55,7,5,latter,ORD,10:47,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MQ,01/11/2020,"3,946.00",N247NN,15:00,15,5,latter,ORD,17:48,...,early,21:19,41.00,21,5,30,19,18,21,2.00
2,MQ,01/18/2020,"3,946.00",N265NN,15:00,15,5,latter,ORD,17:48,...,early,21:19,-2.00,21,5,30,19,18,21,1.00
3,MQ,01/25/2020,"3,946.00",N281NN,15:00,15,5,latter,ORD,17:48,...,early,21:19,-1.00,21,5,30,19,18,21,1.00
4,MQ,02/01/2020,"3,946.00",N283NN,15:00,15,5,latter,ORD,17:48,...,early,21:19,-12.00,21,5,30,19,18,21,0.00


7251

In [16]:
merged_df.head(20)

,Carrier_Code_x,Date,Flight_Number_x,Tail_Number_x,Scheduled departure time_x,dep_hour_x,dep_day_x,dep_order_x,Origin_Airport,Scheduled Arrival Time_x,...,dep_order_y,Scheduled Arrival Time_y,Arrival Delay (Minutes)_y,arr_hour_y,arr_day_y,dep_min_y,arr_min_y,dep_hours_y,arr_hours_y,arr_status_y
0,MQ,01/04/2020,"3,580.00",N240NN,7:55,7,5,latter,ORD,10:47,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MQ,01/11/2020,"3,946.00",N247NN,15:00,15,5,latter,ORD,17:48,...,early,21:19,41.00,21,5,30,19,18,21,2.00
2,MQ,01/18/2020,"3,946.00",N265NN,15:00,15,5,latter,ORD,17:48,...,early,21:19,-2.00,21,5,30,19,18,21,1.00
3,MQ,01/25/2020,"3,946.00",N281NN,15:00,15,5,latter,ORD,17:48,...,early,21:19,-1.00,21,5,30,19,18,21,1.00
4,MQ,02/01/2020,"3,946.00",N283NN,15:00,15,5,latter,ORD,17:48,...,early,21:19,-12.00,21,5,30,19,18,21,0.00
5,MQ,02/08/2020,"3,946.00",N274NN,15:00,15,5,latter,ORD,17:48,...,early,21:19,10.00,21,5,30,19,18,21,2.00
6,MQ,04/06/2020,"3,618.00",NaN,19:29,19,0,latter,ORD,22:16,...,early,20:48,0.00,20,0,0,48,18,20,1.00
7,MQ,04/07/2020,"3,946.00",N287NN,15:06,15,1,latter,ORD,17:53,...,early,20:48,-7.00,20,1,0,48,18,20,0.00
8,MQ,04/08/2020,"3,946.00",NaN,15:06,15,2,latter,ORD,17:53,...,early,20:48,-19.00,20,2,0,48,18,20,0.00
9,MQ,04/09/2020,"3,946.00",NaN,15:06,15,3,latter,ORD,17:53,...,early,20:48,-21.00,20,3,0,48,18,20,0.00


In [17]:
len(merged_df)

7251

In [18]:
merged_df.columns

Index(['Carrier_Code_x', 'Date', 'Flight_Number_x', 'Tail_Number_x',
       'Scheduled departure time_x', 'dep_hour_x', 'dep_day_x', 'dep_order_x',
       'Origin_Airport', 'Scheduled Arrival Time_x',
       'Arrival Delay (Minutes)_x', 'arr_hour_x', 'arr_day_x', 'dep_min_x',
       'arr_min_x', 'dep_hours_x', 'arr_hours_x', 'arr_status_x',
       'Carrier_Code_y', 'Flight_Number_y', 'Tail_Number_y',
       'Scheduled departure time_y', 'dep_hour_y', 'dep_day_y', 'dep_order_y',
       'Scheduled Arrival Time_y', 'Arrival Delay (Minutes)_y', 'arr_hour_y',
       'arr_day_y', 'dep_min_y', 'arr_min_y', 'dep_hours_y', 'arr_hours_y',
       'arr_status_y'],
      dtype='object')

In [19]:
merged_df['hour_diff']= merged_df['dep_hour_x'] - merged_df['dep_hour_y']
merged_df1 = merged_df[merged_df['hour_diff'] == 0 ]
merged_df1 = merged_df1[merged_df1['dep_min_x'] > merged_df1['dep_min_y']]
merged_df2 = merged_df[(merged_df['hour_diff'] >=1) & (merged_df['hour_diff'] <= 2)]
merged_df = pd.concat([merged_df1,merged_df2])
len(merged_df)

893

In [20]:
merged_df.head()

,Carrier_Code_x,Date,Flight_Number_x,Tail_Number_x,Scheduled departure time_x,dep_hour_x,dep_day_x,dep_order_x,Origin_Airport,Scheduled Arrival Time_x,...,Scheduled Arrival Time_y,Arrival Delay (Minutes)_y,arr_hour_y,arr_day_y,dep_min_y,arr_min_y,dep_hours_y,arr_hours_y,arr_status_y,hour_diff
869,MQ,07/04/2023,"3,402.00",N298FR,18:30,18,1,latter,ORD,21:25,...,21:14,19.00,21,1,20,14,18,21,2.00,0
870,MQ,07/05/2023,"3,402.00",N768RD,18:36,18,2,latter,ORD,21:32,...,21:14,104.00,21,2,20,14,18,21,2.00,0
873,MQ,07/06/2023,"3,402.00",N634RW,18:36,18,3,latter,ORD,21:32,...,21:14,-21.00,21,3,20,14,18,21,0.00,0
884,MQ,07/10/2023,"3,402.00",N228NN,18:36,18,0,latter,ORD,21:32,...,21:14,162.00,21,0,20,14,18,21,2.00,0
887,MQ,07/11/2023,"3,402.00",N449YX,18:36,18,1,latter,ORD,21:32,...,21:14,5.00,21,1,20,14,18,21,1.00,0


In [21]:
# merged_df['dep_year_x'].value_counts()

In [22]:
merged_df["Carrier_Code_x"] = np.where(merged_df["Carrier_Code_x"] == "MQ", "AA", merged_df["Carrier_Code_x"])
merged_df["Carrier_Code_x"] = np.where(merged_df["Carrier_Code_x"] == "9E", "DL", merged_df["Carrier_Code_x"])

In [23]:
merged_df.head()

,Carrier_Code_x,Date,Flight_Number_x,Tail_Number_x,Scheduled departure time_x,dep_hour_x,dep_day_x,dep_order_x,Origin_Airport,Scheduled Arrival Time_x,...,Scheduled Arrival Time_y,Arrival Delay (Minutes)_y,arr_hour_y,arr_day_y,dep_min_y,arr_min_y,dep_hours_y,arr_hours_y,arr_status_y,hour_diff
869,AA,07/04/2023,"3,402.00",N298FR,18:30,18,1,latter,ORD,21:25,...,21:14,19.00,21,1,20,14,18,21,2.00,0
870,AA,07/05/2023,"3,402.00",N768RD,18:36,18,2,latter,ORD,21:32,...,21:14,104.00,21,2,20,14,18,21,2.00,0
873,AA,07/06/2023,"3,402.00",N634RW,18:36,18,3,latter,ORD,21:32,...,21:14,-21.00,21,3,20,14,18,21,0.00,0
884,AA,07/10/2023,"3,402.00",N228NN,18:36,18,0,latter,ORD,21:32,...,21:14,162.00,21,0,20,14,18,21,2.00,0
887,AA,07/11/2023,"3,402.00",N449YX,18:36,18,1,latter,ORD,21:32,...,21:14,5.00,21,1,20,14,18,21,1.00,0


In [24]:
merged_df.columns

Index(['Carrier_Code_x', 'Date', 'Flight_Number_x', 'Tail_Number_x',
       'Scheduled departure time_x', 'dep_hour_x', 'dep_day_x', 'dep_order_x',
       'Origin_Airport', 'Scheduled Arrival Time_x',
       'Arrival Delay (Minutes)_x', 'arr_hour_x', 'arr_day_x', 'dep_min_x',
       'arr_min_x', 'dep_hours_x', 'arr_hours_x', 'arr_status_x',
       'Carrier_Code_y', 'Flight_Number_y', 'Tail_Number_y',
       'Scheduled departure time_y', 'dep_hour_y', 'dep_day_y', 'dep_order_y',
       'Scheduled Arrival Time_y', 'Arrival Delay (Minutes)_y', 'arr_hour_y',
       'arr_day_y', 'dep_min_y', 'arr_min_y', 'dep_hours_y', 'arr_hours_y',
       'arr_status_y', 'hour_diff'],
      dtype='object')

In [25]:
merged_df.drop(columns=['Carrier_Code_y', 'Flight_Number_y',
       'Tail_Number_y', 'Scheduled departure time_y', 'dep_hour_y',
       'dep_day_y', 'dep_order_y', 'Scheduled Arrival Time_y',
       'Arrival Delay (Minutes)_y', 'arr_hour_y', 'arr_day_y', 'dep_min_y',
       'arr_min_y', 'dep_hours_y',
       'arr_hours_y'], inplace=True)


In [26]:
merged_df.columns = merged_df.columns.str.replace('_x', '')
merged_df.rename(columns={'arr_status':'arr_status_x'}, inplace=True)

merged_df.head()
merged_df.columns

,Carrier_Code,Date,Flight_Number,Tail_Number,Scheduled departure time,dep_hour,dep_day,dep_order,Origin_Airport,Scheduled Arrival Time,Arrival Delay (Minutes),arr_hour,arr_day,dep_min,arr_min,dep_hours,arr_hours,arr_status_x,arr_status_y,hour_diff
869,AA,07/04/2023,"3,402.00",N298FR,18:30,18,1,latter,ORD,21:25,1.00,21,1,30,25,18,21,1,2.00,0
870,AA,07/05/2023,"3,402.00",N768RD,18:36,18,2,latter,ORD,21:32,0.00,21,2,36,32,18,21,1,2.00,0
873,AA,07/06/2023,"3,402.00",N634RW,18:36,18,3,latter,ORD,21:32,-8.00,21,3,36,32,18,21,0,0.00,0
884,AA,07/10/2023,"3,402.00",N228NN,18:36,18,0,latter,ORD,21:32,63.00,21,0,36,32,18,21,2,2.00,0
887,AA,07/11/2023,"3,402.00",N449YX,18:36,18,1,latter,ORD,21:32,53.00,21,1,36,32,18,21,2,1.00,0


Index(['Carrier_Code', 'Date', 'Flight_Number', 'Tail_Number',
       'Scheduled departure time', 'dep_hour', 'dep_day', 'dep_order',
       'Origin_Airport', 'Scheduled Arrival Time', 'Arrival Delay (Minutes)',
       'arr_hour', 'arr_day', 'dep_min', 'arr_min', 'dep_hours', 'arr_hours',
       'arr_status_x', 'arr_status_y', 'hour_diff'],
      dtype='object')

In [27]:
# merged_df.to_csv('flight_data\\all_lat_ear.csv')

In [28]:
# Read and process weather data files for each airport
jfk_weather_data = pd.read_csv('weather_data/JFK_weather_data_hourly_processed.csv')
syr_weather_data = pd.read_csv('weather_data/SYR_weather_data_hourly_processed.csv')
ord_weather_data = pd.read_csv('weather_data/ORD_weather_data_hourly_processed.csv')
mco_weather_data = pd.read_csv('weather_data/MCO_weather_data_hourly_processed.csv')

# Combine weather data for all airports
weather_dfs = [jfk_weather_data, ord_weather_data, mco_weather_data]
weather_data = pd.concat(weather_dfs, axis=0)
weather_data['dep_hours'] = weather_data['dep_hours'].astype('object')
syr_weather_data['arr_hours'] = syr_weather_data['arr_hours'].astype('object')
weather_data.head()
syr_weather_data.head()
weather_data.dtypes
syr_weather_data.dtypes




,dep_azimuth,dep_clouds,dep_dewpt,dep_elev_angle,dep_h_angle,dep_precip,dep_pres,dep_revision_status,dep_rh,dep_snow,dep_temp,dep_vis,dep_weather.description,dep_weather.code,dep_wind_dir,dep_wind_gust_spd,dep_wind_spd,Date,dep_hours,Origin_Airport
0,261.20,100.00,3.80,-26.20,NaN,0.00,"1,002.00",final,88.00,0.00,5.60,16.00,Overcast clouds,804,220.00,6.00,3.60,12/31/2019,19,JFK
1,270.50,100.00,3.90,-37.50,NaN,0.25,"1,003.00",final,85.00,0.00,6.30,16.00,Overcast clouds,804,220.00,6.80,5.37,12/31/2019,20,JFK
2,281.40,100.00,3.70,-48.80,NaN,0.00,"1,003.00",final,82.00,0.00,6.50,16.00,Overcast clouds,804,255.00,8.00,7.45,12/31/2019,21,JFK
3,296.30,100.00,1.60,-59.60,NaN,0.00,"1,002.00",final,73.00,0.00,6.10,16.00,Overcast clouds,804,270.00,9.30,6.70,12/31/2019,22,JFK
4,320.80,100.00,0.70,-68.60,NaN,0.00,"1,003.00",final,69.00,0.00,6.00,16.00,Overcast clouds,804,270.00,10.40,7.45,12/31/2019,23,JFK


,arr_azimuth,arr_clouds,arr_dewpt,arr_elev_angle,arr_h_angle,arr_precip,arr_pres,arr_revision_status,arr_rh,arr_snow,arr_temp,arr_vis,arr_weather.description,arr_weather.code,arr_wind_dir,arr_wind_gust_spd,arr_wind_spd,Date,arr_hours
0,260.90,100.00,-2.30,-24.90,NaN,0.00,987.00,final,78.00,0.00,1.10,14.00,Overcast clouds,804,240.00,9.60,4.59,12/31/2019,19
1,270.70,100.00,-3.00,-35.80,NaN,0.00,987.00,final,77.00,0.00,0.60,16.00,Overcast clouds,804,240.00,9.60,4.59,12/31/2019,20
2,282.10,100.00,-4.00,-46.60,NaN,0.00,986.00,final,71.00,0.00,0.60,16.00,Overcast clouds,804,240.00,10.40,5.70,12/31/2019,21
3,297.00,100.00,-4.40,-56.90,NaN,0.00,987.00,final,69.00,0.00,0.60,16.00,Overcast clouds,804,250.00,10.40,6.20,12/31/2019,22
4,319.80,100.00,-4.40,-65.60,NaN,0.00,986.00,final,69.00,0.00,0.60,14.00,Overcast clouds,804,250.00,10.80,6.20,12/31/2019,23


dep_azimuth                float64
dep_clouds                 float64
dep_dewpt                  float64
dep_elev_angle             float64
dep_h_angle                float64
dep_precip                 float64
dep_pres                   float64
dep_revision_status         object
dep_rh                     float64
dep_snow                   float64
dep_temp                   float64
dep_vis                    float64
dep_weather.description     object
dep_weather.code             int64
dep_wind_dir               float64
dep_wind_gust_spd          float64
dep_wind_spd               float64
Date                        object
dep_hours                   object
Origin_Airport              object
dtype: object

arr_azimuth                float64
arr_clouds                 float64
arr_dewpt                  float64
arr_elev_angle             float64
arr_h_angle                float64
arr_precip                 float64
arr_pres                   float64
arr_revision_status         object
arr_rh                     float64
arr_snow                   float64
arr_temp                   float64
arr_vis                    float64
arr_weather.description     object
arr_weather.code             int64
arr_wind_dir               float64
arr_wind_gust_spd          float64
arr_wind_spd               float64
Date                        object
arr_hours                   object
dtype: object

In [29]:
# Define merging logic based on airport code
merged_df = pd.merge(merged_df, syr_weather_data, how='left', on=['Date', 'arr_hours'])
merged_df.head()



,Carrier_Code,Date,Flight_Number,Tail_Number,Scheduled departure time,dep_hour,dep_day,dep_order,Origin_Airport,Scheduled Arrival Time,...,arr_revision_status,arr_rh,arr_snow,arr_temp,arr_vis,arr_weather.description,arr_weather.code,arr_wind_dir,arr_wind_gust_spd,arr_wind_spd
0,AA,07/04/2023,"3,402.00",N298FR,18:30,18,1,latter,ORD,21:25,...,final,68.00,0.00,24.40,13.00,Broken clouds,803,325.00,2.00,1.20
1,AA,07/05/2023,"3,402.00",N768RD,18:36,18,2,latter,ORD,21:32,...,final,62.00,0.00,25.60,16.00,Scattered clouds,802,70.00,0.80,0.80
2,AA,07/06/2023,"3,402.00",N634RW,18:36,18,3,latter,ORD,21:32,...,final,62.00,0.00,26.70,16.00,Overcast clouds,804,300.00,3.20,2.60
3,AA,07/10/2023,"3,402.00",N228NN,18:36,18,0,latter,ORD,21:32,...,final,86.00,0.00,17.80,16.00,Overcast clouds,804,190.00,2.80,1.50
4,AA,07/11/2023,"3,402.00",N449YX,18:36,18,1,latter,ORD,21:32,...,final,61.00,0.00,25.00,16.00,Overcast clouds,804,320.00,2.80,1.50


In [30]:
merged_df = pd.merge(merged_df, weather_data, how='left', on=['Origin_Airport', 'Date', 'dep_hours'])


merged_df.head()

,Carrier_Code,Date,Flight_Number,Tail_Number,Scheduled departure time,dep_hour,dep_day,dep_order,Origin_Airport,Scheduled Arrival Time,...,dep_revision_status,dep_rh,dep_snow,dep_temp,dep_vis,dep_weather.description,dep_weather.code,dep_wind_dir,dep_wind_gust_spd,dep_wind_spd
0,AA,07/04/2023,"3,402.00",N298FR,18:30,18,1,latter,ORD,21:25,...,final,54.00,0.00,30.10,16.00,Overcast clouds,804,280.00,5.20,3.45
1,AA,07/05/2023,"3,402.00",N768RD,18:36,18,2,latter,ORD,21:32,...,final,79.00,0.00,26.10,16.00,Thunderstorm with heavy rain,202,290.00,10.30,5.70
2,AA,07/06/2023,"3,402.00",N634RW,18:36,18,3,latter,ORD,21:32,...,final,55.00,0.00,24.70,16.00,Overcast clouds,804,65.00,6.00,3.97
3,AA,07/10/2023,"3,402.00",N228NN,18:36,18,0,latter,ORD,21:32,...,final,43.00,0.00,29.10,16.00,Broken clouds,803,210.00,9.60,5.30
4,AA,07/11/2023,"3,402.00",N449YX,18:36,18,1,latter,ORD,21:32,...,final,68.00,0.00,21.80,16.00,Broken clouds,803,70.00,6.40,3.10


In [31]:
# merged_df.to_csv('flight_data\\all_lat_ear.csv')

In [32]:
merged_df.drop(columns= ['Date', 'Flight_Number', 'Tail_Number',
       'Scheduled departure time','Carrier_Code',
       'dep_order', 'Scheduled Arrival Time',
       'Arrival Delay (Minutes)','hour_diff','arr_weather.description','dep_weather.description'],inplace = True)

merged_df.dtypes

dep_hour                object
dep_day                 object
Origin_Airport          object
arr_hour                object
arr_day                 object
dep_min                 object
arr_min                 object
dep_hours               object
arr_hours               object
arr_status_x             int32
arr_status_y           float64
arr_azimuth            float64
arr_clouds             float64
arr_dewpt              float64
arr_elev_angle         float64
arr_h_angle            float64
arr_precip             float64
arr_pres               float64
arr_revision_status     object
arr_rh                 float64
arr_snow               float64
arr_temp               float64
arr_vis                float64
arr_weather.code         int64
arr_wind_dir           float64
arr_wind_gust_spd      float64
arr_wind_spd           float64
dep_azimuth            float64
dep_clouds             float64
dep_dewpt              float64
dep_elev_angle         float64
dep_h_angle            float64
dep_prec

In [33]:
merged_df['arr_status_y'] = merged_df['arr_status_y'].astype('int64').astype('object')
merged_df.head()
merged_df.dtypes
su_data = merged_df

,dep_hour,dep_day,Origin_Airport,arr_hour,arr_day,dep_min,arr_min,dep_hours,arr_hours,arr_status_x,...,dep_pres,dep_revision_status,dep_rh,dep_snow,dep_temp,dep_vis,dep_weather.code,dep_wind_dir,dep_wind_gust_spd,dep_wind_spd
0,18,1,ORD,21,1,30,25,18,21,1,...,989.00,final,54.00,0.00,30.10,16.00,804,280.00,5.20,3.45
1,18,2,ORD,21,2,36,32,18,21,1,...,986.00,final,79.00,0.00,26.10,16.00,202,290.00,10.30,5.70
2,18,3,ORD,21,3,36,32,18,21,0,...,989.00,final,55.00,0.00,24.70,16.00,804,65.00,6.00,3.97
3,18,0,ORD,21,0,36,32,18,21,2,...,987.00,final,43.00,0.00,29.10,16.00,803,210.00,9.60,5.30
4,18,1,ORD,21,1,36,32,18,21,2,...,989.00,final,68.00,0.00,21.80,16.00,803,70.00,6.40,3.10


dep_hour                object
dep_day                 object
Origin_Airport          object
arr_hour                object
arr_day                 object
dep_min                 object
arr_min                 object
dep_hours               object
arr_hours               object
arr_status_x             int32
arr_status_y            object
arr_azimuth            float64
arr_clouds             float64
arr_dewpt              float64
arr_elev_angle         float64
arr_h_angle            float64
arr_precip             float64
arr_pres               float64
arr_revision_status     object
arr_rh                 float64
arr_snow               float64
arr_temp               float64
arr_vis                float64
arr_weather.code         int64
arr_wind_dir           float64
arr_wind_gust_spd      float64
arr_wind_spd           float64
dep_azimuth            float64
dep_clouds             float64
dep_dewpt              float64
dep_elev_angle         float64
dep_h_angle            float64
dep_prec

In [34]:
# su_data = sub_data.drop(columns = ['dep_order'])
su_data.head()

,dep_hour,dep_day,Origin_Airport,arr_hour,arr_day,dep_min,arr_min,dep_hours,arr_hours,arr_status_x,...,dep_pres,dep_revision_status,dep_rh,dep_snow,dep_temp,dep_vis,dep_weather.code,dep_wind_dir,dep_wind_gust_spd,dep_wind_spd
0,18,1,ORD,21,1,30,25,18,21,1,...,989.00,final,54.00,0.00,30.10,16.00,804,280.00,5.20,3.45
1,18,2,ORD,21,2,36,32,18,21,1,...,986.00,final,79.00,0.00,26.10,16.00,202,290.00,10.30,5.70
2,18,3,ORD,21,3,36,32,18,21,0,...,989.00,final,55.00,0.00,24.70,16.00,804,65.00,6.00,3.97
3,18,0,ORD,21,0,36,32,18,21,2,...,987.00,final,43.00,0.00,29.10,16.00,803,210.00,9.60,5.30
4,18,1,ORD,21,1,36,32,18,21,2,...,989.00,final,68.00,0.00,21.80,16.00,803,70.00,6.40,3.10


In [35]:
su_data.columns
su_data.head()

Index(['dep_hour', 'dep_day', 'Origin_Airport', 'arr_hour', 'arr_day',
       'dep_min', 'arr_min', 'dep_hours', 'arr_hours', 'arr_status_x',
       'arr_status_y', 'arr_azimuth', 'arr_clouds', 'arr_dewpt',
       'arr_elev_angle', 'arr_h_angle', 'arr_precip', 'arr_pres',
       'arr_revision_status', 'arr_rh', 'arr_snow', 'arr_temp', 'arr_vis',
       'arr_weather.code', 'arr_wind_dir', 'arr_wind_gust_spd', 'arr_wind_spd',
       'dep_azimuth', 'dep_clouds', 'dep_dewpt', 'dep_elev_angle',
       'dep_h_angle', 'dep_precip', 'dep_pres', 'dep_revision_status',
       'dep_rh', 'dep_snow', 'dep_temp', 'dep_vis', 'dep_weather.code',
       'dep_wind_dir', 'dep_wind_gust_spd', 'dep_wind_spd'],
      dtype='object')

,dep_hour,dep_day,Origin_Airport,arr_hour,arr_day,dep_min,arr_min,dep_hours,arr_hours,arr_status_x,...,dep_pres,dep_revision_status,dep_rh,dep_snow,dep_temp,dep_vis,dep_weather.code,dep_wind_dir,dep_wind_gust_spd,dep_wind_spd
0,18,1,ORD,21,1,30,25,18,21,1,...,989.00,final,54.00,0.00,30.10,16.00,804,280.00,5.20,3.45
1,18,2,ORD,21,2,36,32,18,21,1,...,986.00,final,79.00,0.00,26.10,16.00,202,290.00,10.30,5.70
2,18,3,ORD,21,3,36,32,18,21,0,...,989.00,final,55.00,0.00,24.70,16.00,804,65.00,6.00,3.97
3,18,0,ORD,21,0,36,32,18,21,2,...,987.00,final,43.00,0.00,29.10,16.00,803,210.00,9.60,5.30
4,18,1,ORD,21,1,36,32,18,21,2,...,989.00,final,68.00,0.00,21.80,16.00,803,70.00,6.40,3.10


In [36]:

su_data['dep_hour'] = pd.Categorical(su_data['dep_hour'], categories=[i for i in range(24)])
su_data['dep_day'] = pd.Categorical(su_data['dep_day'], categories=[i for i in range(7)])
su_data['dep_min'] = pd.Categorical(su_data['dep_min'], categories=[i for i in range(60)])
su_data['arr_hour'] = pd.Categorical(su_data['arr_hour'], categories=[i for i in range(24)])
su_data['arr_day'] = pd.Categorical(su_data['arr_day'], categories=[i for i in range(7)])
su_data['arr_min'] = pd.Categorical(su_data['arr_min'], categories=[i for i in range(60)])
#su_data['Carrier_Code'] = pd.Categorical(su_data['Carrier_Code'], categories=['AA', 'DL', 'B6'])
su_data['Origin_Airport'] = pd.Categorical(su_data['Origin_Airport'], categories=['ORD', 'JFK', 'MCO'])
su_data['arr_weather.code'] = pd.Categorical(su_data['arr_weather.code'], categories=[200,201,202,230,231,232,233,300,301,302,500,501,502,511,520,522,600,602,610,611,612,621,622,633,700,711,721,731,741,751,800,801,802,803,804,900])
su_data['dep_weather.code'] = pd.Categorical(su_data['dep_weather.code'], categories=[200,201,202,230,231,232,233,300,301,302,500,501,502,511,520,522,600,602,610,611,612,621,622,633,700,711,721,731,741,751,800,801,802,803,804,900])
su_data.drop(columns=['dep_hours', 'arr_hours'],inplace = True)
su_data.columns

Index(['dep_hour', 'dep_day', 'Origin_Airport', 'arr_hour', 'arr_day',
       'dep_min', 'arr_min', 'arr_status_x', 'arr_status_y', 'arr_azimuth',
       'arr_clouds', 'arr_dewpt', 'arr_elev_angle', 'arr_h_angle',
       'arr_precip', 'arr_pres', 'arr_revision_status', 'arr_rh', 'arr_snow',
       'arr_temp', 'arr_vis', 'arr_weather.code', 'arr_wind_dir',
       'arr_wind_gust_spd', 'arr_wind_spd', 'dep_azimuth', 'dep_clouds',
       'dep_dewpt', 'dep_elev_angle', 'dep_h_angle', 'dep_precip', 'dep_pres',
       'dep_revision_status', 'dep_rh', 'dep_snow', 'dep_temp', 'dep_vis',
       'dep_weather.code', 'dep_wind_dir', 'dep_wind_gust_spd',
       'dep_wind_spd'],
      dtype='object')

In [37]:
su_data.isna().sum()

dep_hour                 0
dep_day                  0
Origin_Airport           0
arr_hour                 0
arr_day                  0
dep_min                  0
arr_min                  0
arr_status_x             0
arr_status_y             0
arr_azimuth              0
arr_clouds               0
arr_dewpt                0
arr_elev_angle           0
arr_h_angle            893
arr_precip               0
arr_pres                 0
arr_revision_status      0
arr_rh                   0
arr_snow                 0
arr_temp                 0
arr_vis                  0
arr_weather.code         8
arr_wind_dir             0
arr_wind_gust_spd        0
arr_wind_spd             0
dep_azimuth              0
dep_clouds               0
dep_dewpt                0
dep_elev_angle           0
dep_h_angle            893
dep_precip               0
dep_pres                 0
dep_revision_status      0
dep_rh                   0
dep_snow                 0
dep_temp                 0
dep_vis                  0
d

In [38]:
su_data.drop(columns=['arr_h_angle','dep_h_angle', 'arr_azimuth', 'dep_azimuth', 'arr_elev_angle', 'dep_elev_angle', 'arr_revision_status', 'dep_revision_status'],inplace = True)
su_data.dropna(inplace=True)

# sub_data.drop(columns= ['Carrier_Code','Date','Flight_Number','Tail_Number','arr_azimuth',
#                        'Scheduled departure time','Scheduled Arrival Time','dep_order', 
#                         'arr_hours', 'dep_hours',
#                         'arr_weather.description', 'dep_weather.description','arr_delay','Departure delay (Minutes)',
#                        'arr_elev_angle','arr_h_angle','arr_revision_status',
#                        'dep_elev_angle','dep_h_angle','dep_revision_status','dep_azimuth'],inplace = True)


In [39]:
su_data.columns

Index(['dep_hour', 'dep_day', 'Origin_Airport', 'arr_hour', 'arr_day',
       'dep_min', 'arr_min', 'arr_status_x', 'arr_status_y', 'arr_clouds',
       'arr_dewpt', 'arr_precip', 'arr_pres', 'arr_rh', 'arr_snow', 'arr_temp',
       'arr_vis', 'arr_weather.code', 'arr_wind_dir', 'arr_wind_gust_spd',
       'arr_wind_spd', 'dep_clouds', 'dep_dewpt', 'dep_precip', 'dep_pres',
       'dep_rh', 'dep_snow', 'dep_temp', 'dep_vis', 'dep_weather.code',
       'dep_wind_dir', 'dep_wind_gust_spd', 'dep_wind_spd'],
      dtype='object')

In [40]:
su_data = pd.get_dummies(su_data, drop_first = True)
#su_data['arr_status'] = su_data['arr_status'].astype('object')
su_data.head()
su_data.dtypes

,arr_status_x,arr_clouds,arr_dewpt,arr_precip,arr_pres,arr_rh,arr_snow,arr_temp,arr_vis,arr_wind_dir,...,dep_weather.code_721,dep_weather.code_731,dep_weather.code_741,dep_weather.code_751,dep_weather.code_800,dep_weather.code_801,dep_weather.code_802,dep_weather.code_803,dep_weather.code_804,dep_weather.code_900
0,1,50.00,18.10,0.00,999.00,68.00,0.00,24.40,13.00,325.00,...,False,False,False,False,False,False,False,False,True,False
1,1,25.00,17.80,0.00,999.00,62.00,0.00,25.60,16.00,70.00,...,False,False,False,False,False,False,False,False,False,False
2,0,87.00,18.80,0.00,995.00,62.00,0.00,26.70,16.00,300.00,...,False,False,False,False,False,False,False,False,True,False
3,2,87.00,15.40,0.00,995.00,86.00,0.00,17.80,16.00,190.00,...,False,False,False,False,False,False,False,True,False,False
4,2,87.00,17.00,0.00,996.00,61.00,0.00,25.00,16.00,320.00,...,False,False,False,False,False,False,False,True,False,False


arr_status_x              int32
arr_clouds              float64
arr_dewpt               float64
arr_precip              float64
arr_pres                float64
                         ...   
dep_weather.code_801       bool
dep_weather.code_802       bool
dep_weather.code_803       bool
dep_weather.code_804       bool
dep_weather.code_900       bool
Length: 273, dtype: object

In [41]:
X_train, X_test, y_train, y_test = train_test_split(su_data.drop(columns = ['arr_status_x']), su_data['arr_status_x'], test_size=0.2, stratify = su_data['arr_status_x'], random_state=50)
# In the above split the stratify = y essentially makes sure the fractions of the classification is maintained
X_train
X_test
y_train.dtypes
y_test

,arr_clouds,arr_dewpt,arr_precip,arr_pres,arr_rh,arr_snow,arr_temp,arr_vis,arr_wind_dir,arr_wind_gust_spd,...,dep_weather.code_721,dep_weather.code_731,dep_weather.code_741,dep_weather.code_751,dep_weather.code_800,dep_weather.code_801,dep_weather.code_802,dep_weather.code_803,dep_weather.code_804,dep_weather.code_900
333,50.00,-5.70,0.00,995.00,71.00,0.00,-1.10,16.00,90.00,2.80,...,False,False,False,False,False,False,False,True,False,False
178,87.00,5.40,0.00,"1,009.00",49.00,0.00,16.10,16.00,300.00,7.20,...,False,False,False,False,False,False,False,False,True,False
281,50.00,17.20,0.00,995.00,87.00,0.00,19.40,16.00,220.00,3.20,...,False,False,False,False,False,False,False,True,False,False
297,87.00,13.80,0.00,998.00,83.00,0.00,16.70,16.00,190.00,6.00,...,False,False,False,False,False,False,True,False,False,False
278,25.00,18.30,0.00,"1,006.00",81.00,0.00,21.70,16.00,35.00,2.80,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
506,50.00,-7.40,0.00,"1,007.00",49.00,0.00,2.20,16.00,90.00,4.40,...,False,False,False,False,True,False,False,False,False,False
370,100.00,19.40,0.00,988.00,84.00,0.00,22.20,16.00,285.00,8.40,...,False,False,False,False,False,False,False,False,True,False
440,100.00,11.70,0.00,985.00,100.00,0.00,11.70,10.00,250.00,7.60,...,False,False,False,False,False,False,False,True,False,False
694,100.00,11.10,0.25,988.00,83.00,0.00,13.90,11.00,140.00,10.00,...,False,False,False,False,False,False,False,False,True,False


,arr_clouds,arr_dewpt,arr_precip,arr_pres,arr_rh,arr_snow,arr_temp,arr_vis,arr_wind_dir,arr_wind_gust_spd,...,dep_weather.code_721,dep_weather.code_731,dep_weather.code_741,dep_weather.code_751,dep_weather.code_800,dep_weather.code_801,dep_weather.code_802,dep_weather.code_803,dep_weather.code_804,dep_weather.code_900
711,100.00,8.10,0.00,"1,006.00",66.00,0.00,14.40,16.00,50.00,4.00,...,False,False,False,False,False,False,False,False,True,False
668,25.00,8.90,0.00,"1,007.00",80.00,0.00,12.20,16.00,135.00,2.40,...,False,False,False,False,False,False,False,True,False,False
821,100.00,1.00,0.00,999.00,85.00,0.00,3.30,10.00,300.00,9.20,...,False,False,False,False,False,False,False,False,True,False
422,100.00,5.50,0.00,"1,004.00",92.00,0.00,6.70,8.00,80.00,7.60,...,False,False,False,False,False,False,False,True,False,False
639,100.00,18.30,0.00,996.00,81.00,0.00,21.70,16.00,240.00,4.40,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
837,87.00,-12.30,0.00,"1,008.00",39.00,0.00,0.00,16.00,170.00,9.80,...,False,False,False,False,False,False,True,False,False,False
644,87.00,11.10,0.00,998.00,72.00,0.00,16.10,16.00,340.00,8.00,...,False,False,False,False,False,False,True,False,False,False
208,50.00,-5.00,0.00,"1,002.00",72.00,0.00,-0.60,16.00,210.00,9.30,...,False,False,False,False,False,False,False,True,False,False
465,100.00,-4.40,0.00,"1,010.00",78.00,0.00,-1.10,16.00,160.00,2.80,...,False,False,False,False,False,False,False,False,True,False


dtype('int32')

711    1
668    2
821    1
422    1
639    0
      ..
837    1
644    0
208    2
465    0
835    2
Name: arr_status_x, Length: 177, dtype: int32

In [42]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = pd.DataFrame(sc.fit_transform(X_train), columns = X_train.columns, index = X_train.index)
X_test = pd.DataFrame(sc.transform(X_test), columns = X_test.columns, index = X_test.index)
X_train
X_test
y_train
y_test

,arr_clouds,arr_dewpt,arr_precip,arr_pres,arr_rh,arr_snow,arr_temp,arr_vis,arr_wind_dir,arr_wind_gust_spd,...,dep_weather.code_721,dep_weather.code_731,dep_weather.code_741,dep_weather.code_751,dep_weather.code_800,dep_weather.code_801,dep_weather.code_802,dep_weather.code_803,dep_weather.code_804,dep_weather.code_900
333,-0.77,-1.38,-0.20,-0.67,0.17,-0.08,-1.51,0.40,-1.21,-1.15,...,-0.09,0.00,-0.17,0.00,-0.21,-0.16,-0.48,2.31,-1.00,0.00
178,0.39,-0.05,-0.20,1.10,-1.06,-0.08,0.45,0.40,1.17,0.17,...,-0.09,0.00,-0.17,0.00,-0.21,-0.16,-0.48,-0.43,1.00,0.00
281,-0.77,1.37,-0.20,-0.67,1.06,-0.08,0.83,0.40,0.26,-1.03,...,-0.09,0.00,-0.17,0.00,-0.21,-0.16,-0.48,2.31,-1.00,0.00
297,0.39,0.96,-0.20,-0.29,0.84,-0.08,0.52,0.40,-0.08,-0.19,...,-0.09,0.00,-0.17,0.00,-0.21,-0.16,2.07,-0.43,-1.00,0.00
278,-1.56,1.50,-0.20,0.72,0.73,-0.08,1.09,0.40,-1.83,-1.15,...,-0.09,0.00,-0.17,0.00,-0.21,-0.16,-0.48,2.31,-1.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
506,-0.77,-1.58,-0.20,0.84,-1.06,-0.08,-1.14,0.40,-1.21,-0.67,...,-0.09,0.00,-0.17,0.00,4.67,-0.16,-0.48,-0.43,-1.00,0.00
370,0.80,1.63,-0.20,-1.56,0.89,-0.08,1.15,0.40,1.00,0.53,...,-0.09,0.00,-0.17,0.00,-0.21,-0.16,-0.48,-0.43,1.00,0.00
440,0.80,0.71,-0.20,-1.94,1.79,-0.08,-0.05,-1.59,0.60,0.29,...,-0.09,0.00,-0.17,0.00,-0.21,-0.16,-0.48,2.31,-1.00,0.00
694,0.80,0.64,0.06,-1.56,0.84,-0.08,0.20,-1.26,-0.65,1.01,...,-0.09,0.00,-0.17,0.00,-0.21,-0.16,-0.48,-0.43,1.00,0.00


,arr_clouds,arr_dewpt,arr_precip,arr_pres,arr_rh,arr_snow,arr_temp,arr_vis,arr_wind_dir,arr_wind_gust_spd,...,dep_weather.code_721,dep_weather.code_731,dep_weather.code_741,dep_weather.code_751,dep_weather.code_800,dep_weather.code_801,dep_weather.code_802,dep_weather.code_803,dep_weather.code_804,dep_weather.code_900
711,0.80,0.28,-0.20,0.72,-0.11,-0.08,0.25,0.40,-1.66,-0.79,...,-0.09,0.00,-0.17,0.00,-0.21,-0.16,-0.48,-0.43,1.00,0.00
668,-1.56,0.37,-0.20,0.84,0.67,-0.08,0.00,0.40,-0.70,-1.27,...,-0.09,0.00,-0.17,0.00,-0.21,-0.16,-0.48,2.31,-1.00,0.00
821,0.80,-0.58,-0.20,-0.17,0.95,-0.08,-1.01,-1.59,1.17,0.77,...,-0.09,0.00,-0.17,0.00,-0.21,-0.16,-0.48,-0.43,1.00,0.00
422,0.80,-0.04,-0.20,0.46,1.34,-0.08,-0.62,-2.26,-1.32,0.29,...,-0.09,0.00,-0.17,0.00,-0.21,-0.16,-0.48,2.31,-1.00,0.00
639,0.80,1.50,-0.20,-0.55,0.73,-0.08,1.09,0.40,0.49,-0.67,...,-0.09,0.00,-0.17,0.00,-0.21,-0.16,-0.48,2.31,-1.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
837,0.39,-2.17,-0.20,0.97,-1.62,-0.08,-1.39,0.40,-0.31,0.95,...,-0.09,0.00,-0.17,0.00,-0.21,-0.16,2.07,-0.43,-1.00,0.00
644,0.39,0.64,-0.20,-0.29,0.22,-0.08,0.45,0.40,1.62,0.41,...,-0.09,0.00,-0.17,0.00,-0.21,-0.16,2.07,-0.43,-1.00,0.00
208,-0.77,-1.30,-0.20,0.21,0.22,-0.08,-1.46,0.40,0.15,0.80,...,-0.09,0.00,-0.17,0.00,-0.21,-0.16,-0.48,2.31,-1.00,0.00
465,0.80,-1.22,-0.20,1.22,0.56,-0.08,-1.51,0.40,-0.42,-1.15,...,-0.09,0.00,-0.17,0.00,-0.21,-0.16,-0.48,-0.43,1.00,0.00


333    2
178    2
281    2
297    2
278    2
      ..
506    0
370    2
440    0
694    1
86     0
Name: arr_status_x, Length: 708, dtype: int32

711    1
668    2
821    1
422    1
639    0
      ..
837    1
644    0
208    2
465    0
835    2
Name: arr_status_x, Length: 177, dtype: int32

In [43]:
#model = LogisticRegression(fit_intercept = True, solver='newton-cg', multi_class = 'multinomial', penalty = 'none', max_iter = 1000)
# model = LogisticRegression(fit_intercept = True, solver='lbfgs', multi_class = 'ovr', penalty = 'none', max_iter = 1000)
model = LogisticRegression(fit_intercept = True, solver='lbfgs', multi_class = 'multinomial', penalty = None, max_iter = 1000)
# model = LogisticRegression(fit_intercept = True, solver='newton-cg', multi_class = 'multinomial', penalty = 'none', max_iter = 10000)

# While using multiclass case do multi_class = 'ovr' or 'auto'; can also try other solvers
# While doing regularization, use penalty = 'l2' and also C = 10.0 (need to try other values too)

model.fit(X_train, y_train) 

# The following gives the mean accuracy on the given data and labels
model.score(X_train, y_train) 

# This is the coefficient Beta_1, ..., Beta_7
model.coef_

# This is the coefficient Beta_0
model.intercept_

LogisticRegression(max_iter=1000, multi_class='multinomial', penalty=None)

0.6892655367231638

array([[-1.77557688e-01,  1.51864226e+00, -6.29568731e-01,
        -3.95301039e-01, -9.75742501e-01,  9.46889332e-01,
        -1.86002736e+00, -4.89936587e-02, -3.38835364e-02,
        -1.03907265e-01,  1.12027914e-02, -6.05578439e-01,
        -1.12503171e+00, -1.75253835e+00,  8.59351548e-01,
         6.10982083e-01,  3.97996187e-01,  1.66001415e+00,
         5.50443293e-02, -1.71543113e-01, -2.41120570e-01,
         2.05758609e-01,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         6.49795097e-01,  1.75192267e-01,  2.71908508e-01,
         2.51692037e-01, -3.55612212e-02,  7.46853289e-01,
         6.69205794e-02,  0.00000000e+00,  0.00000000e+00,
         7.10313204e-02, -3.22204974e-01,  1.01214859e+00,
         2.07882414e-01, -1.30470952e+00, -8.33147418e-01,
         4.69222264e-02,  2.54892182e-02, -4.30858480e-02,
        -5.44719844e-02, -8.94664758e-03,  3.42075583e-0

array([ 0.87449327, -0.56666747, -0.3078258 ])

In [44]:
model.score(X_test,y_test)

0.4915254237288136

In [45]:
from sklearn.tree import DecisionTreeClassifier
# from sklearn.tree import DecisionTreeRegressor
clf = DecisionTreeClassifier(random_state=50, min_samples_leaf = 3)

clf = clf.fit(X_train, y_train) 
clf.score(X_train, y_train)

clf.feature_importances_
test_output = pd.DataFrame(clf.predict(X_test), index = X_test.index, columns = ['pred_arr_status'])
test_output.head()
clf.score(X_test, y_test)

0.8771186440677966

array([0.00957746, 0.05933293, 0.        , 0.05291329, 0.08992641,
       0.        , 0.07039763, 0.01448766, 0.06991406, 0.04754783,
       0.01465176, 0.03030252, 0.06420566, 0.        , 0.11865059,
       0.02428676, 0.        , 0.06658488, 0.00605907, 0.01487149,
       0.05016564, 0.03398052, 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.01194158, 0.        , 0.        , 0.        , 0.00377745,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.00894885, 0.00151477, 0.        ,
       0.        , 0.00393839, 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.00545632, 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.     

,pred_arr_status
711,2
668,2
821,1
422,2
639,0


0.423728813559322

In [46]:

from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(random_state=50, min_samples_split = 12, min_samples_leaf = 6, max_depth = 4, n_estimators = 300)

gb = gb.fit(X_train, y_train) 
gb.score(X_train, y_train) 

1.0

In [47]:
gb.score(X_test,y_test)

0.480225988700565

In [48]:
pred_data2 = pd.read_csv('pred_data2.csv')
pred_data2.head()
pred_data2.dtypes

,Unnamed: 0,dep_hour,dep_day,Origin_Airport,arr_hour,arr_day,dep_min,arr_min,arr_clouds,arr_dewpt,...,dep_precip,dep_pres,dep_rh,dep_snow,dep_temp,dep_vis,dep_weather.code,dep_wind_dir,dep_wind_gust_spd,dep_wind_spd
0,1,19,4,ORD,22,4,59,52,66,7.20,...,0.00,996.50,37,0,12.10,24.00,803,280,11.10,7.60
1,3,14,4,JFK,16,4,55,21,86,7.90,...,0.00,"1,019.00",59,0,12.30,24.00,804,120,7.30,5.60
2,5,13,4,MCO,16,4,35,25,86,7.90,...,0.00,"1,013.50",45,0,30.90,24.00,801,270,3.20,2.40
3,7,19,5,ORD,22,5,59,52,23,-2.40,...,0.00,994.50,41,0,9.50,24.13,804,300,6.00,4.00
4,9,14,5,JFK,16,5,55,21,58,-1.60,...,0.00,"1,014.00",42,0,16.50,24.00,803,270,6.00,4.40


Unnamed: 0             int64
dep_hour               int64
dep_day                int64
Origin_Airport        object
arr_hour               int64
arr_day                int64
dep_min                int64
arr_min                int64
arr_clouds             int64
arr_dewpt            float64
arr_precip           float64
arr_pres             float64
arr_rh                 int64
arr_snow               int64
arr_temp             float64
arr_vis              float64
arr_weather.code       int64
arr_wind_dir           int64
arr_wind_gust_spd    float64
arr_wind_spd         float64
dep_clouds             int64
dep_dewpt            float64
dep_precip           float64
dep_pres             float64
dep_rh                 int64
dep_snow               int64
dep_temp             float64
dep_vis              float64
dep_weather.code       int64
dep_wind_dir           int64
dep_wind_gust_spd    float64
dep_wind_spd         float64
dtype: object

In [49]:

pred_data2['dep_min'] = pred_data2['dep_min'].astype('object')
pred_data2['arr_min'] = pred_data2['arr_min'].astype('object')
pred_data2['dep_hour'] = pred_data2['dep_hour'].astype('object')
pred_data2['dep_day'] = pred_data2['dep_day'].astype('object')
pred_data2['arr_hour'] = pred_data2['arr_hour'].astype('object')
pred_data2['arr_day'] = pred_data2['arr_day'].astype('object')
pred_data2['dep_weather.code'] = pred_data2['dep_weather.code'].astype('object')
pred_data2['arr_weather.code'] = pred_data2['arr_weather.code'].astype('object')
pred_data2.drop(columns=['Unnamed: 0'],inplace=True)

pred_data2.dtypes

dep_hour              object
dep_day               object
Origin_Airport        object
arr_hour              object
arr_day               object
dep_min               object
arr_min               object
arr_clouds             int64
arr_dewpt            float64
arr_precip           float64
arr_pres             float64
arr_rh                 int64
arr_snow               int64
arr_temp             float64
arr_vis              float64
arr_weather.code      object
arr_wind_dir           int64
arr_wind_gust_spd    float64
arr_wind_spd         float64
dep_clouds             int64
dep_dewpt            float64
dep_precip           float64
dep_pres             float64
dep_rh                 int64
dep_snow               int64
dep_temp             float64
dep_vis              float64
dep_weather.code      object
dep_wind_dir           int64
dep_wind_gust_spd    float64
dep_wind_spd         float64
dtype: object

In [50]:
pred_data2['dep_hour'] = pd.Categorical(pred_data2['dep_hour'], categories=[i for i in range(24)])
pred_data2['dep_day'] = pd.Categorical(pred_data2['dep_day'], categories=[i for i in range(7)])
pred_data2['dep_min'] = pd.Categorical(pred_data2['dep_min'], categories=[i for i in range(60)])
pred_data2['arr_hour'] = pd.Categorical(pred_data2['arr_hour'], categories=[i for i in range(24)])
pred_data2['arr_day'] = pd.Categorical(pred_data2['arr_day'], categories=[i for i in range(7)])
pred_data2['arr_min'] = pd.Categorical(pred_data2['arr_min'], categories=[i for i in range(60)])
#su_data['Carrier_Code'] = pd.Categorical(su_data['Carrier_Code'], categories=['AA', 'UA', 'DL', 'B6', 'WN'])
pred_data2['Origin_Airport'] = pd.Categorical(pred_data2['Origin_Airport'], categories=['ORD', 'JFK', 'MCO'])

pred_data2['arr_weather.code'] = pd.Categorical(pred_data2['arr_weather.code'], categories=[200,201,202,230,231,232,233,300,301,302,500,501,502,511,520,522,600,602,610,611,612,621,622,633,700,711,721,731,741,751,800,801,802,803,804,900])
pred_data2['dep_weather.code'] = pd.Categorical(pred_data2['dep_weather.code'], categories=[200,201,202,230,231,232,233,300,301,302,500,501,502,511,520,522,600,602,610,611,612,621,622,633,700,711,721,731,741,751,800,801,802,803,804,900])



In [51]:
len(pred_data2)

12

In [52]:
my_data = {'arr_status_y': [0, 1, 2]}
df = pd.DataFrame(data = my_data)

In [53]:
df.head()

,arr_status_y
0,0
1,1
2,2


In [54]:
pred_data2 =pred_data2.merge(df,how = "cross")
len(pred_data2)

36

In [55]:
pred_data2['arr_status_y'] = pd.Categorical(pred_data2['arr_status_y'], categories=[0,1,2])

In [56]:
pred_data2 = pred_data2[['dep_hour', 'dep_day', 'Origin_Airport', 'arr_hour', 'arr_day',
       'dep_min', 'arr_min', 'arr_status_y', 'arr_clouds',
       'arr_dewpt', 'arr_precip', 'arr_pres', 'arr_rh', 'arr_snow', 'arr_temp',
       'arr_vis', 'arr_weather.code', 'arr_wind_dir', 'arr_wind_gust_spd',
       'arr_wind_spd', 'dep_clouds', 'dep_dewpt', 'dep_precip', 'dep_pres',
       'dep_rh', 'dep_snow', 'dep_temp', 'dep_vis',
       'dep_weather.code', 'dep_wind_dir', 'dep_wind_gust_spd',
       'dep_wind_spd']]

In [57]:
pred_data2 = pd.get_dummies(pred_data2, drop_first = True)
#su_data['arr_status'] = su_data['arr_status'].astype('object')
pred_data2.head()
pred_data2.dtypes
pred_data2.columns

,arr_clouds,arr_dewpt,arr_precip,arr_pres,arr_rh,arr_snow,arr_temp,arr_vis,arr_wind_dir,arr_wind_gust_spd,...,dep_weather.code_721,dep_weather.code_731,dep_weather.code_741,dep_weather.code_751,dep_weather.code_800,dep_weather.code_801,dep_weather.code_802,dep_weather.code_803,dep_weather.code_804,dep_weather.code_900
0,66,7.20,0.25,999.00,73,0,11.90,19.49,260,8.10,...,False,False,False,False,False,False,False,True,False,False
1,66,7.20,0.25,999.00,73,0,11.90,19.49,260,8.10,...,False,False,False,False,False,False,False,True,False,False
2,66,7.20,0.25,999.00,73,0,11.90,19.49,260,8.10,...,False,False,False,False,False,False,False,True,False,False
3,86,7.90,0.76,996.00,59,0,15.90,20.80,180,13.20,...,False,False,False,False,False,False,False,False,True,False
4,86,7.90,0.76,996.00,59,0,15.90,20.80,180,13.20,...,False,False,False,False,False,False,False,False,True,False


arr_clouds                int64
arr_dewpt               float64
arr_precip              float64
arr_pres                float64
arr_rh                    int64
                         ...   
dep_weather.code_801       bool
dep_weather.code_802       bool
dep_weather.code_803       bool
dep_weather.code_804       bool
dep_weather.code_900       bool
Length: 272, dtype: object

Index(['arr_clouds', 'arr_dewpt', 'arr_precip', 'arr_pres', 'arr_rh',
       'arr_snow', 'arr_temp', 'arr_vis', 'arr_wind_dir', 'arr_wind_gust_spd',
       ...
       'dep_weather.code_721', 'dep_weather.code_731', 'dep_weather.code_741',
       'dep_weather.code_751', 'dep_weather.code_800', 'dep_weather.code_801',
       'dep_weather.code_802', 'dep_weather.code_803', 'dep_weather.code_804',
       'dep_weather.code_900'],
      dtype='object', length=272)

In [58]:
from sklearn.preprocessing import StandardScaler
#sc = StandardScaler()
X_test = pd.DataFrame(sc.transform(pred_data2), columns = pred_data2.columns, index = pred_data2.index)


In [59]:
linear_reg_output = pd.DataFrame(gb.predict(X_test), index = X_test.index, columns = ['pred_arr_status'])
#linear_reg_output = linear_reg_output.merge(y_test, left_index = True, right_index = True)
linear_reg_output.head(50)
# mean_absolute_error = abs(linear_reg_output['pred_delay'] - linear_reg_output['arr_delay']).mean()
# print('Mean absolute error is ')
# print(mean_absolute_error)
# maer = mean_absolute_error/linear_reg_output['pred_delay'].mean()
# print('Mean absolute error ratio is ')
# print(maer)
#model.score(X_test, y_test)

,pred_arr_status
0,0
1,0
2,0
3,0
4,0
5,0
6,0
7,0
8,0
9,0
